# 🔬 02 — Statistical Tests

This notebook demonstrates statistical analysis of survey data including:

- **Reliability analysis** (Cronbach's alpha)
- **Group comparisons** (t-tests, ANOVA, non-parametric)
- **Correlation analysis** (Pearson, Spearman, Kendall)
- **Chi-square tests** of independence
- **Factor analysis** (EFA)
- **Proportion tests**
- **Scale score computation**

---

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from survey_toolkit import (
    SurveyLoader,
    SurveyCleaner,
    SurveyStats,
    generate_sample_survey,
    detect_column_types,
    compute_scale_scores,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_style("whitegrid")

print("✅ Setup complete")

## 2. Load & Clean Data

In [ ]:
# Generate fresh data (or load existing)
df = generate_sample_survey(
    n_respondents=500,
    n_likert_items=10,
    random_state=42,
)

# Detect column types
types = detect_column_types(df)
likert_cols = types["likert"]
print(f"Likert columns: {likert_cols}")

# Clean
cleaner = SurveyCleaner(df)
clean_df = (
    cleaner
    .remove_speeders("duration_seconds", min_seconds=60)
    .remove_straightliners(likert_cols, threshold=0.95)
    .handle_missing(strategy="median", columns=likert_cols)
    .get_clean_data()
)

print(f"\n📋 Clean dataset: {clean_df.shape[0]} respondents, {clean_df.shape[1]} columns")

In [ ]:
# Initialize stats engine
stats = SurveyStats(clean_df)

## 3. Reliability Analysis (Cronbach's Alpha)

Cronbach's alpha measures **internal consistency** — how closely related
a set of items are as a group. It's essential for validating survey scales.

| Alpha | Interpretation |
|-------|---------------|
| ≥ 0.9 | Excellent |
| ≥ 0.8 | Good |
| ≥ 0.7 | Acceptable |
| ≥ 0.6 | Questionable |
| ≥ 0.5 | Poor |
| < 0.5 | Unacceptable |

In [ ]:
# Full scale reliability
alpha_result = stats.cronbachs_alpha(likert_cols)

print("=" * 60)
print("CRONBACH'S ALPHA — FULL SCALE")
print("=" * 60)
print(f"  Alpha:          {alpha_result['alpha']}")
print(f"  Interpretation: {alpha_result['interpretation']}")
print(f"  Items:          {alpha_result['n_items']}")
print(f"  Valid N:        {alpha_result['n_valid']}")

In [ ]:
# Item-total correlations
print("\n📊 Item-Total Correlations:")
print("-" * 40)
itc = alpha_result["item_total_correlations"]
for item, corr in sorted(itc.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(corr * 20) if corr > 0 else ""
    print(f"  {item:>5}: {corr:>7.4f}  {bar}")

In [ ]:
# Alpha if item deleted
print("\n📊 Alpha if Item Deleted:")
print("-" * 40)
aid = alpha_result["alpha_if_deleted"]
for item, a in sorted(aid.items()):
    flag = "  ⬆️ REMOVE?" if a > alpha_result["alpha"] else ""
    print(f"  {item:>5}: {a:>7.4f}{flag}")

In [ ]:
# Visualize item diagnostics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Item-total correlations
items = list(itc.keys())
corrs = [itc[item] for item in items]
colors = ["#1a9850" if c >= 0.3 else "#d73027" for c in corrs]
axes[0].barh(items, corrs, color=colors, edgecolor="white")
axes[0].axvline(x=0.3, color="red", linestyle="--", alpha=0.5, label="Threshold (0.3)")
axes[0].set_xlabel("Corrected Item-Total Correlation")
axes[0].set_title("Item-Total Correlations")
axes[0].legend()

# Alpha if deleted
alphas = [aid[item] for item in items]
colors2 = ["#d73027" if a > alpha_result["alpha"] else "#1a9850" for a in alphas]
axes[1].barh(items, alphas, color=colors2, edgecolor="white")
axes[1].axvline(x=alpha_result["alpha"], color="blue", linestyle="--",
                alpha=0.5, label=f"Current α = {alpha_result['alpha']:.4f}")
axes[1].set_xlabel("Cronbach's Alpha")
axes[1].set_title("Alpha if Item Deleted")
axes[1].legend()

plt.tight_layout()
plt.savefig("../outputs/figures/reliability_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

### Sub-scale Reliability

If items cluster into constructs, check reliability per construct.

In [ ]:
# Split into two sub-scales
subscale_a = likert_cols[:5]
subscale_b = likert_cols[5:]

alpha_a = stats.cronbachs_alpha(subscale_a)
alpha_b = stats.cronbachs_alpha(subscale_b)

print(f"Sub-scale A ({subscale_a}): α = {alpha_a['alpha']} ({alpha_a['interpretation']})")
print(f"Sub-scale B ({subscale_b}): α = {alpha_b['alpha']} ({alpha_b['interpretation']})")

## 4. Group Comparisons

Compare survey responses across demographic groups.
The toolkit **automatically selects the correct test**:

| Groups | Normal Data | Non-Normal Data |
|--------|------------|-----------------|
| 2 | Independent t-test | Mann-Whitney U |
| 3+ | One-way ANOVA | Kruskal-Wallis H |

### 4.1 Two-Group Comparison (Gender)

In [ ]:
# Filter to two groups for clean comparison
df_two = clean_df[clean_df["gender"].isin(["Male", "Female"])].copy()
stats_two = SurveyStats(df_two)

comparison_2g = stats_two.compare_groups("q1", "gender", test="auto")

print("=" * 60)
print("TWO-GROUP COMPARISON: Q1 by Gender")
print("=" * 60)
print(f"  Test:           {comparison_2g['test']}")
print(f"  Statistic:      {comparison_2g['statistic']}")
print(f"  p-value:        {comparison_2g['p_value']}")
print(f"  Effect size:    {comparison_2g['effect_size']} ({comparison_2g['effect_size_name']})")
print(f"  Significant:    {'✅ Yes' if comparison_2g['significant'] else '❌ No'}")
print(f"  Normality:      {'Assumed' if comparison_2g['normality_assumed'] else 'Not assumed'}")
print(f"  Equal variance: {'Yes' if comparison_2g['equal_variance'] else 'No'}")

print("\n  Group Statistics:")
for name, gs in comparison_2g["group_stats"].items():
    print(f"    {name}: M={gs['mean']}, Mdn={gs['median']}, SD={gs['std']}, n={gs['n']}")

### 4.2 Multi-Group Comparison (Age Group)

In [ ]:
comparison_mg = stats.compare_groups("q1", "age_group", test="auto")

print("=" * 60)
print("MULTI-GROUP COMPARISON: Q1 by Age Group")
print("=" * 60)
print(f"  Test:        {comparison_mg['test']}")
print(f"  Statistic:   {comparison_mg['statistic']}")
print(f"  p-value:     {comparison_mg['p_value']}")
print(f"  Effect size: {comparison_mg['effect_size']} ({comparison_mg['effect_size_name']})")
print(f"  Significant: {'✅ Yes' if comparison_mg['significant'] else '❌ No'}")

print("\n  Group Statistics:")
for name, gs in comparison_mg["group_stats"].items():
    print(f"    {name:>8}: M={gs['mean']:.2f}, SD={gs['std']:.2f}, n={gs['n']}")

# Post-hoc results
if "posthoc" in comparison_mg:
    print(f"\n  Post-Hoc (Tukey HSD):")
    print(comparison_mg["posthoc"])

### 4.3 Compare All Likert Items Across Groups

In [ ]:
comparison_results = []
for col in likert_cols:
    result = stats.compare_groups(col, "age_group", test="auto")
    comparison_results.append({
        "variable": col,
        "test": result["test"],
        "statistic": result["statistic"],
        "p_value": result["p_value"],
        "effect_size": result["effect_size"],
        "significant": result["significant"],
    })

comp_df = pd.DataFrame(comparison_results)
print("\n📊 All Items by Age Group:")
print(comp_df.to_string(index=False))

# Highlight significant results
sig_count = comp_df["significant"].sum()
print(f"\n✅ {sig_count}/{len(likert_cols)} items showed significant group differences")

## 5. Correlation Analysis

In [ ]:
corr_result = stats.correlation_matrix(likert_cols, method="pearson")

print(f"Method: {corr_result['method']}")
print(f"N: {corr_result['n']}")
print(f"\n📊 Correlation Matrix:")
display(corr_result["correlation_matrix"].round(3))

In [ ]:
# Significant correlations
print(f"\n🔗 Significant Correlations (p < 0.05):")
print("-" * 60)
for pair in corr_result["significant_pairs"]:
    strength = (
        "Strong" if abs(pair["correlation"]) >= 0.7 else
        "Moderate" if abs(pair["correlation"]) >= 0.4 else
        "Weak"
    )
    print(f"  {pair['var1']} ↔ {pair['var2']}: "
          f"r = {pair['correlation']:>7.4f} "
          f"(p = {pair['p_value']:.6f}) [{strength}]")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_result["correlation_matrix"], dtype=bool))
sns.heatmap(
    corr_result["correlation_matrix"],
    mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
)
ax.set_title("Pearson Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/correlation_matrix_stats.png", dpi=150, bbox_inches="tight")
plt.show()

### Spearman Correlation (Non-parametric)

In [ ]:
spearman_result = stats.correlation_matrix(likert_cols, method="spearman")

print(f"\n📊 Spearman vs Pearson (diagonal comparison):")
for col in likert_cols[:5]:
    for col2 in likert_cols[:5]:
        if col < col2:
            p = corr_result["correlation_matrix"].loc[col, col2]
            s = spearman_result["correlation_matrix"].loc[col, col2]
            diff = abs(p - s)
            flag = " ⚠️" if diff > 0.05 else ""
            print(f"  {col}-{col2}: Pearson={p:.3f}, Spearman={s:.3f}, Δ={diff:.3f}{flag}")

## 6. Chi-Square Tests

Test independence between categorical variables.

In [ ]:
chi2_result = stats.chi_square_test("age_group", "gender")

print("=" * 60)
print("CHI-SQUARE TEST: Age Group × Gender")
print("=" * 60)
print(f"  χ²:             {chi2_result['chi2']}")
print(f"  p-value:        {chi2_result['p_value']}")
print(f"  df:             {chi2_result['dof']}")
print(f"  Cramér's V:     {chi2_result['cramers_v']}")
print(f"  Effect size:    {chi2_result['effect_size_interpretation']}")
print(f"  Significant:    {'✅ Yes' if chi2_result['significant'] else '❌ No'}")

if "warning" in chi2_result:
    print(f"\n  ⚠️ {chi2_result['warning']}")

In [ ]:
print("\n📊 Observed Frequencies:")
display(chi2_result["contingency_table"])

In [ ]:
print("\n📊 Expected Frequencies:")
display(chi2_result["expected_frequencies"])

In [ ]:
# Test satisfaction by demographics
if "satisfaction_group" in clean_df.columns:
    chi2_sat = stats.chi_square_test("satisfaction_group", "age_group")
    print(f"\nSatisfaction × Age Group: χ²={chi2_sat['chi2']:.4f}, "
          f"p={chi2_sat['p_value']:.6f}, V={chi2_sat['cramers_v']:.4f}")

    chi2_sat2 = stats.chi_square_test("satisfaction_group", "gender")
    print(f"Satisfaction × Gender:    χ²={chi2_sat2['chi2']:.4f}, "
          f"p={chi2_sat2['p_value']:.6f}, V={chi2_sat2['cramers_v']:.4f}")

## 7. Factor Analysis (EFA)

Exploratory Factor Analysis identifies latent constructs
underlying the survey items.

### 7.1 Adequacy Testing & Factor Extraction

In [ ]:
fa_result = stats.factor_analysis(
    likert_cols,
    n_factors=None,    # Auto-detect using Kaiser criterion
    rotation="varimax",
    method="ml",
)

print("=" * 60)
print("EXPLORATORY FACTOR ANALYSIS")
print("=" * 60)
print(f"\n  Adequacy Tests:")
print(f"    Bartlett's χ²:  {fa_result['bartlett_chi_sq']} (p = {fa_result['bartlett_p']})")
print(f"    Bartlett sig:   {'✅ Yes' if fa_result['bartlett_significant'] else '❌ No'}")
print(f"    KMO:            {fa_result['kmo']} ({fa_result['kmo_interpretation']})")
print(f"    KMO adequate:   {'✅ Yes' if fa_result['kmo_adequate'] else '❌ No'}")
print(f"\n  Extraction:")
print(f"    Method:         {fa_result['method']}")
print(f"    Rotation:       {fa_result['rotation']}")
print(f"    Factors:        {fa_result['n_factors']}")
print(f"    Observations:   {fa_result['n_observations']}")

In [ ]:
print("\n📊 Factor Loadings:")
display(fa_result["loadings"].round(4))

In [ ]:
# Highlight loadings > 0.4
print("\n📊 Significant Loadings (> 0.4):")
loadings = fa_result["loadings"]
for item in loadings.index:
    row = loadings.loc[item]
    sig_loadings = row[row.abs() > 0.4]
    if not sig_loadings.empty:
        loading_str = ", ".join([f"{f}: {v:.3f}" for f, v in sig_loadings.items()])
        print(f"  {item}: {loading_str}")

In [ ]:
# Item assignments
print("\n📊 Item → Factor Assignments:")
for item, factor in fa_result["item_assignments"].items():
    loading = fa_result["loadings"].loc[item, factor]
    print(f"  {item} → {factor} (loading = {loading:.3f})")

In [ ]:
# Variance explained
print("\n📊 Variance Explained:")
ve = fa_result["variance_explained"]
for i, (eigen, pct, cum) in enumerate(
    zip(ve["eigenvalues"], ve["per_factor"], ve["cumulative"])
):
    print(f"  Factor {i+1}: eigenvalue={eigen:.4f}, "
          f"variance={pct:.1%}, cumulative={cum:.1%}")

In [ ]:
# Communalities
print("\n📊 Communalities:")
for item, comm in fa_result["communalities"].items():
    bar = "█" * int(comm * 20)
    print(f"  {item}: {comm:.4f} {bar}")

In [ ]:
# Visualize factor loadings
loadings = fa_result["loadings"]
n_factors = loadings.shape[1]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(loadings))
width = 0.8 / n_factors

for i, factor in enumerate(loadings.columns):
    ax.bar(x + i * width, loadings[factor], width,
           label=factor, alpha=0.8)

ax.set_xticks(x + width * (n_factors - 1) / 2)
ax.set_xticklabels(loadings.index, rotation=45)
ax.set_ylabel("Loading")
ax.set_title("Factor Loadings by Item")
ax.legend()
ax.axhline(y=0.4, color="red", linestyle="--", alpha=0.5, label="Threshold")
ax.axhline(y=-0.4, color="red", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("../outputs/figures/factor_loadings.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Forced 2-Factor Solution

In [ ]:
fa_2 = stats.factor_analysis(likert_cols, n_factors=2, rotation="varimax")
print(f"2-Factor solution — Cumulative variance: {fa_2['variance_explained']['cumulative'][-1]:.1%}")
display(fa_2["loadings"].round(3))

## 8. Scale Score Computation

In [ ]:
# Build construct map from factor assignments
construct_map = {}
for item, factor in fa_result["item_assignments"].items():
    if factor not in construct_map:
        construct_map[factor] = []
    construct_map[factor].append(item)

print("📊 Construct Map:")
for construct, items in construct_map.items():
    print(f"  {construct}: {items}")

In [ ]:
# Compute composite scores
scores = compute_scale_scores(clean_df, construct_map, method="mean")
print(f"\n📊 Scale Scores ({len(scores.columns)} constructs):")
display(scores.describe().round(3))

In [ ]:
# Correlate scale scores
if len(scores.columns) >= 2:
    score_corr = scores.corr().round(3)
    print("\n📊 Inter-Scale Correlations:")
    display(score_corr)

## 9. Proportion Tests

In [ ]:
prop_result = stats.proportion_test("gender", "Male")
print(f"One-sample proportion (Male):")
print(f"  Proportion: {prop_result['proportion']}")
print(f"  p-value:    {prop_result['p_value']}")
print(f"  Significant: {'✅' if prop_result['significant'] else '❌'} "
      f"(different from 50%)")

## 10. Grouped Descriptive Statistics

In [ ]:
desc = stats.descriptives_by_group(likert_cols[:5], "age_group")
display(desc)

## 11. Summary of All Results

In [ ]:
all_results = stats.get_all_results()
print(f"\n📊 Analyses completed: {len(all_results)}")
for name in all_results.keys():
    print(f"  ✅ {name}")